In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import time
import pytz
from datetime import datetime
import numpy as np

mt5.initialize()


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 100)
    rates_frame = pd.DataFrame(rates)

    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    
    rates_frame['sma']= rates_frame['close'].rolling(window=50).mean()

    return rates_frame


def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)


def run(symbol):
    check = 0
    lot = 0.1
    buy_check = 0
    sell_check = 0
    buy_up = 0
    sell_up = 0
    order_time = 0

    buy = 1
    sell = 0
    old = 0
 
    print(symbol)
    hour_passed = True

    while True:
        a = get_values(symbol)
        if a.iloc[-2].close != old:
            
            if a.iloc[-2].close <= a.iloc[-2].sma and  a.iloc[-2].open >= a.iloc[-2].sma and sell_check == 0:   

                result_sell = Action(symbol, lot, sell)
                print(f"Symbol-->{symbol} ||| Type-->Sell  ||| Ticket_No-->{result_sell.order}")
                old = a.iloc[-2].close

                sell_check = 1

                buy_check = 0

            elif a.iloc[-2].close >= a.iloc[-2].sma and a.iloc[-2].open <= a.iloc[-2].sma and buy_check == 0:

                result_buy = Action(symbol, lot, buy)
                print(f"Symbol-->{symbol} ||| Type-->Buy ||| Ticket_No-->{result_buy.order} ||| result_comment-->{result_buy.comment}")
                buy_check = 1
                old = a.iloc[-2].close

                sell_check = 0

        if sell_check==1:
            pp = mt5.positions_get(ticket=result_sell.order)[0].profit
            if pp<= -10.0 and sell_check == 1:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0
            if pp >= 10.0 and sell_check == 1:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close


                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0

        if buy_check==1:
            pp = mt5.positions_get(ticket=result_buy.order)[0].profit
            if  pp<= -10.0 and buy_check == 1:
                result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                if result_buy.comment == "Requote":
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                buy_check = 0
            if pp >= 10.0 and buy_check == 1:
                result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                if result_buy.comment == "Requote":
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                buy_check = 0

#             hour_passed = True
        time.sleep(1)

for symbol in ['GBPUSD']:
    run(symbol)

GBPUSD
Symbol-->GBPUSD ||| Type-->Sell  ||| Ticket_No-->5193290838
Symbol-->GBPUSD ||| Type-->Buy ||| Ticket_No-->5193345016 ||| result_comment-->Request executed
Symbol-->GBPUSD ||| Type-->Sell  ||| Ticket_No-->5193389511
Close  Symbol-->GBPUSD ||| Type-->Sell ||| result_comment-->Request executed
Symbol-->GBPUSD ||| Type-->Buy ||| Ticket_No-->5193507404 ||| result_comment-->Request executed
Close  Symbol-->GBPUSD ||| Type-->Buy ||| result_comment-->Request executed
Symbol-->GBPUSD ||| Type-->Buy ||| Ticket_No-->5193520764 ||| result_comment-->Request executed
Close  Symbol-->GBPUSD ||| Type-->Buy ||| result_comment-->Request executed
Symbol-->GBPUSD ||| Type-->Buy ||| Ticket_No-->5193531849 ||| result_comment-->Request executed
Symbol-->GBPUSD ||| Type-->Sell  ||| Ticket_No-->5193542288
Close  Symbol-->GBPUSD ||| Type-->Sell ||| result_comment-->Request executed
Symbol-->GBPUSD ||| Type-->Buy ||| Ticket_No-->5193657347 ||| result_comment-->Request executed
Close  Symbol-->GBPUSD ||| 